In [1]:
# -*- coding: utf-8 -*-
"""
Notebook: Import dữ liệu Bộ luật Hình sự từ CSV vào Neo4j
"""

import pandas as pd
from neo4j import GraphDatabase
import re
from tqdm import tqdm
import os

# ==================================================
# CẤU HÌNH KẾT NỐI NEO4J
# ==================================================
NEO4J_URI = "bolt://localhost:7687"          # Địa chỉ Neo4j
NEO4J_USER = "neo4j"                         # Tên đăng nhập
NEO4J_PASSWORD = "12345678"                  # Mật khẩu

# ==================================================
# ĐỌC DỮ LIỆU CSV
# ==================================================
csv_path = "blhs_enriched.csv"               # Đường dẫn file CSV đã được xử lý
df = pd.read_csv(csv_path, encoding='utf-8-sig')

# Hiển thị thông tin dữ liệu
print(f"Đã đọc {len(df)} dòng.")
print("Các cột:", df.columns.tolist())

# ==================================================
# HÀM XỬ LÝ CHUỖI: TÁCH DANH SÁCH CÁCH NHAU BẰNG DẤU PHẨY
# ==================================================
def split_list_field(field):
    """Trả về danh sách các phần tử được tách bởi dấu phẩy, bỏ qua giá trị NaN, None."""
    if pd.isna(field) or field is None or field == '':
        return []
    # Xử lý trường hợp chuỗi có nhiều dấu phẩy và khoảng trắng
    items = [item.strip() for item in str(field).split(',') if item.strip() and item.strip() != 'nan']
    return items

# ==================================================
# KẾT NỐI ĐẾN NEO4J
# ==================================================
class Neo4jImporter:
    def __init__(self, uri, user, password):
        self.driver = GraphDatabase.driver(uri, auth=(user, password))
        self.session = self.driver.session()

    def close(self):
        self.session.close()
        self.driver.close()

    def create_constraints(self):
        """Tạo các ràng buộc unique cho các node chính."""
        constraints = [
            "CREATE CONSTRAINT IF NOT EXISTS FOR (a:Article) REQUIRE a.id IS UNIQUE",
            "CREATE CONSTRAINT IF NOT EXISTS FOR (cg:CrimeGroup) REQUIRE cg.name IS UNIQUE",
            "CREATE CONSTRAINT IF NOT EXISTS FOR (cc:CrimeCategory) REQUIRE cc.name IS UNIQUE",
            "CREATE CONSTRAINT IF NOT EXISTS FOR (it:IntentType) REQUIRE it.name IS UNIQUE",
            "CREATE CONSTRAINT IF NOT EXISTS FOR (act:Actor) REQUIRE act.name IS UNIQUE",
            "CREATE CONSTRAINT IF NOT EXISTS FOR (vic:Victim) REQUIRE vic.name IS UNIQUE",
            "CREATE CONSTRAINT IF NOT EXISTS FOR (p:Penalty) REQUIRE p.name IS UNIQUE",
            "CREATE CONSTRAINT IF NOT EXISTS FOR (ps:PenaltySupplementary) REQUIRE ps.name IS UNIQUE",
            "CREATE CONSTRAINT IF NOT EXISTS FOR (ac:AggravatingCirc) REQUIRE ac.name IS UNIQUE",
            "CREATE CONSTRAINT IF NOT EXISTS FOR (mc:MitigatingCirc) REQUIRE mc.name IS UNIQUE",
            "CREATE CONSTRAINT IF NOT EXISTS FOR (cond:Condition) REQUIRE cond.name IS UNIQUE",
            "CREATE CONSTRAINT IF NOT EXISTS FOR (ex:Exception) REQUIRE ex.name IS UNIQUE",
            "CREATE CONSTRAINT IF NOT EXISTS FOR (jm:JudicialMeasure) REQUIRE jm.name IS UNIQUE",
        ]
        for cypher in constraints:
            self.session.run(cypher)
        print("Đã tạo ràng buộc.")


    def import_article(self, row):
        """Import một điều luật và các thực thể liên quan."""
        # Tạo node Article
        article_id = row['Số điều']
        if pd.isna(article_id):
            return

        # Tạo câu lệnh MERGE cho Article
        # Chuyển các giá trị sang string an toàn
        def safe_str(val):
            return '' if pd.isna(val) else str(val)

        article_data = {
            'id': str(article_id),
            'title': safe_str(row['Tiêu đề điều']),
            'content': safe_str(row['Nội dung điều']),
            'part': safe_str(row['Phần']),
            'chapter': safe_str(row['Chương']),
            'full_text': safe_str(row['full_text']),
        }
        # Thêm các cột khác nếu cần
        cypher_article = """
        MERGE (a:Article {id: $id})
        SET a.title = $title,
            a.content = $content,
            a.part = $part,
            a.chapter = $chapter,
            a.full_text = $full_text
        RETURN a
        """
        self.session.run(cypher_article, article_data)

        # Xử lý các quan hệ với các node tham chiếu
        # 1. CrimeGroup
        if not pd.isna(row['crime_group']):
            self._create_relationship(row['crime_group'], 'CrimeGroup', 'BELONGS_TO', article_id)

        # 2. CrimeCategory
        if not pd.isna(row['crime_category']):
            self._create_relationship(row['crime_category'], 'CrimeCategory', 'HAS_CATEGORY', article_id)

        # 3. IntentType
        if not pd.isna(row['intent_type']):
            self._create_relationship(row['intent_type'], 'IntentType', 'HAS_INTENT', article_id)

        # 4. Actor (có thể có nhiều giá trị, tách bằng dấu phẩy)
        actors = split_list_field(row['actor'])
        for act in actors:
            self._create_relationship(act, 'Actor', 'HAS_ACTOR', article_id)

        # 5. Victim
        victims = split_list_field(row['victim'])
        for vic in victims:
            self._create_relationship(vic, 'Victim', 'HAS_VICTIM', article_id)

        # 6. Penalty main
        penalties = split_list_field(row['penalty_main'])
        for pen in penalties:
            self._create_relationship(pen, 'Penalty', 'HAS_MAIN_PENALTY', article_id)

        # 7. Penalty supplementary
        supp_penalties = split_list_field(row['penalty_supplementary'])
        for spen in supp_penalties:
            self._create_relationship(spen, 'PenaltySupplementary', 'HAS_SUPP_PENALTY', article_id)

        # 8. Aggravating circumstances
        aggs = split_list_field(row['aggravating_circ'])
        for agg in aggs:
            self._create_relationship(agg, 'AggravatingCirc', 'HAS_AGGRAVATING', article_id)

        # 9. Mitigating circumstances
        mitis = split_list_field(row['mitigating_circ'])
        for mit in mitis:
            self._create_relationship(mit, 'MitigatingCirc', 'HAS_MITIGATING', article_id)

        # 10. Condition
        conds = split_list_field(row['condition'])
        for cond in conds:
            self._create_relationship(cond, 'Condition', 'HAS_CONDITION', article_id)

        # 11. Exception
        excs = split_list_field(row['exception'])
        for exc in excs:
            self._create_relationship(exc, 'Exception', 'HAS_EXCEPTION', article_id)

        # 12. Judicial measure
        jms = split_list_field(row['judicial_measure'])
        for jm in jms:
            self._create_relationship(jm, 'JudicialMeasure', 'HAS_JUDICIAL_MEASURE', article_id)


    def _create_relationship(self, node_name, node_label, rel_type, article_id):
        """Tạo node (nếu chưa tồn tại) và tạo quan hệ với Article."""
        if not node_name:
            return
        # Tạo node nếu chưa có
        cypher_node = f"""
        MERGE (n:{node_label} {{name: $name}})
        RETURN n
        """
        self.session.run(cypher_node, {'name': node_name})

        # Tạo quan hệ
        cypher_rel = f"""
        MATCH (a:Article {{id: $article_id}})
        MATCH (n:{node_label} {{name: $name}})
        MERGE (a)-[:{rel_type}]->(n)
        """
        self.session.run(cypher_rel, {'article_id': article_id, 'name': node_name})


    def import_all(self, df, batch_size=100):
        """Import toàn bộ dữ liệu."""
        total = len(df)
        print(f"Bắt đầu import {total} dòng...")
        for idx, row in tqdm(df.iterrows(), total=total, desc="Importing"):
            self.import_article(row)
            # Có thể commit transaction sau mỗi batch để tăng hiệu suất
            if idx % batch_size == 0:
                # Neo4j Python driver tự động commit sau mỗi session.run, không cần commit thủ công
                pass
        print("Hoàn tất import.")


# ==================================================
# THỰC THI
# ==================================================
if __name__ == "__main__":
    importer = Neo4jImporter(NEO4J_URI, NEO4J_USER, NEO4J_PASSWORD)
    importer.create_constraints()
    importer.import_all(df)
    importer.close()
    print("Đã import xong.")

Đã đọc 3640 dòng.
Các cột: ['Phần', 'Chương', 'Số điều', 'Tiêu đề điều', 'Nội dung điều', 'Số khoản', 'Nội dung khoản', 'Điểm', 'Nội dung điểm', 'full_text', 'crime_group', 'crime_category', 'intent_type', 'actor', 'victim', 'action', 'consequence', 'penalty_main', 'penalty_supplementary', 'aggravating_circ', 'mitigating_circ', 'condition', 'exception', 'judicial_measure']
Đã tạo ràng buộc.
Bắt đầu import 3640 dòng...


Importing: 100%|██████████| 3640/3640 [02:39<00:00, 22.84it/s]

Hoàn tất import.
Đã import xong.
